# 08｜啟動 Flask 服務

在背景執行 Flask；Colab 會顯示代理視窗，Kaggle/Jupyter 會提供代理連結。


In [ ]:
import os
import socket
import sys
import threading
import time
from IPython.display import HTML, display


def _port_is_open(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        return sock.connect_ex(("127.0.0.1", port)) == 0


def launch_server(port=5000):
    """Start Flask once and expose the appropriate notebook link."""
    global flask_thread

    existing = globals().get("flask_thread")
    if existing is not None and existing.is_alive() and _port_is_open(port):
        print(f"Flask 已在連接埠 {port} 執行。")
    elif _port_is_open(port):
        raise RuntimeError(
            f"連接埠 {port} 已被其他程式占用；請重新啟動工作階段，或改用其他 port。"
        )
    else:
        def run_flask():
            app.run(host="0.0.0.0", port=port, threaded=True, use_reloader=False)

        flask_thread = threading.Thread(target=run_flask, daemon=True)
        flask_thread.start()
        for _ in range(50):
            if _port_is_open(port):
                break
            time.sleep(0.1)
        else:
            raise RuntimeError("Flask 服務未能在預期時間內啟動。")
        print(f"Flask 已啟動：http://127.0.0.1:{port}")

    if "google.colab" in sys.modules:
        from google.colab import output
        output.serve_kernel_port_as_window(port)
    elif os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        display(HTML(
            '<p><strong>Flask 已在 Kaggle 核心內啟動。</strong></p>'
            '<p>Kaggle 原生編輯器目前不會直接公開任意核心連接埠。'
            '若要使用 Web 介面，請從 Kaggle 的「Run → Kaggle Jupyter Server」'
            '選擇「Open in Colab」；連上同一個 Kaggle 後端後，再執行本 cell。</p>'
        ))
    else:
        proxy_path = f"/proxy/{port}/"
        display(HTML(
            f'<p><a href="{proxy_path}" target="_blank">'
            f'開啟溺水偵測系統（port {port}）</a></p>'
            '<p>若目前平台未提供 notebook port proxy，請使用平台提供的 '
            'Web Preview / Port Preview 功能開啟相同連接埠。</p>'
        ))
